# Revisão do M1

Este notebook verifica o M1 atual e orienta sua refatoração para responder melhor à RQ1.

**Conclusão principal:** o cálculo atual está correto, mas a métrica não está conceitualmente validada. Ela combina perguntas sobre IA, carreira e projeto usando uma escala sem âncoras explícitas. A refatoração deve substituir o valor único por indicadores separados e interpretáveis.

## De onde vêm os dados do M1?

### Resposta curta

O M1 vem das **respostas dos estudantes a seis perguntas do survey**. Ele não usa dados de Git, transcrições ou avaliações das equipes.

O fluxo é:

1. O data lake reúne 187 submissões de estudantes.
2. Para cada submissão, seis respostas são avaliadas por um LLM. Cada resposta recebe um `ai_dependency_score` inteiro de 0 a 4.
3. Os 1.122 escores resultantes são resumidos por `Semestre`, `temporal_marker` e pergunta.
4. O M1 é a média simples das médias das seis perguntas em cada combinação de semestre e corte.

As seis perguntas tratam de benefício da IA, impacto na carreira em cinco anos, autonomia e dependência de ferramentas, expectativa de carreira, desafios do projeto e sentimento sobre o projeto.

In [48]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

STUDENT_RESPONSES_PATH = PROJECT_ROOT / "data/lake/student_responses.parquet"
STUDENT_NLP_PATH = PROJECT_ROOT / "data/analysis/student_nlp.parquet"
SOURCE_PATH = PROJECT_ROOT / "data/analysis/textual_cut_signals.parquet"
M1_ORIGINAL_PATH = PROJECT_ROOT / "paper_v8/data/m1_ai_dependency_trajectory.csv"

student_responses_df = pd.read_parquet(STUDENT_RESPONSES_PATH)
student_nlp_df = pd.read_parquet(STUDENT_NLP_PATH)
m1_source_df = pd.read_parquet(SOURCE_PATH)
m1_original_df = pd.read_csv(M1_ORIGINAL_PATH, dtype={"Semestre": str})

provenance_df = pd.DataFrame(
    [
        {
            "etapa": "1. Respostas do survey",
            "arquivo": "data/lake/student_responses.parquet",
            "linhas": len(student_responses_df),
            "unidade": "submissão de estudante",
        },
        {
            "etapa": "2. Respostas pontuadas",
            "arquivo": "data/analysis/student_nlp.parquet",
            "linhas": len(student_nlp_df),
            "unidade": "resposta × pergunta",
        },
        {
            "etapa": "3. Resumo por corte",
            "arquivo": "data/analysis/textual_cut_signals.parquet",
            "linhas": len(m1_source_df),
            "unidade": "semestre × corte",
        },
        {
            "etapa": "4. M1 final",
            "arquivo": "paper_v8/data/m1_ai_dependency_trajectory.csv",
            "linhas": len(m1_original_df),
            "unidade": "semestre × corte",
        },
    ]
)
provenance_df

,etapa,arquivo,linhas,unidade
0,1. Respostas do survey,data/lake/student_responses.parquet,187,submissão de estudante
1,2. Respostas pontuadas,data/analysis/student_nlp.parquet,1122,resposta × pergunta
2,3. Resumo por corte,data/analysis/textual_cut_signals.parquet,6,semestre × corte
3,4. M1 final,paper_v8/data/m1_ai_dependency_trajectory.csv,6,semestre × corte


In [49]:
coverage_df = (
    student_nlp_df.groupby(
        ["Semestre", "temporal_marker"],
        as_index=False,
    )
    .agg(
        estudantes=("student_response_id", "nunique"),
        respostas_pontuadas=("ai_dependency_score", "size"),
        perguntas=("question_id", "nunique"),
    )
)
coverage_df

,Semestre,temporal_marker,estudantes,respostas_pontuadas,perguntas
0,2025.2,T1,46,276,6
1,2025.2,T2,41,246,6
2,2025.2,T3,40,240,6
3,2026.1,T1,22,132,6
4,2026.1,T2,20,120,6
5,2026.1,T3,18,108,6


### Como o valor final é produzido

Para cada semestre $s$ e corte $c$, primeiro é calculada a média do `ai_dependency_score` de cada pergunta. Depois, as seis médias recebem o mesmo peso:

$$
M1_{s,c}=\frac{1}{6}\sum_{q=1}^{6}\overline{x}_{q,s,c}
$$

O arquivo `textual_cut_signals.parquet` contém as médias por pergunta. O arquivo `m1_ai_dependency_trajectory.csv` contém somente o resultado final, o semestre, o corte e a quantidade de famílias disponíveis.

**Unidade do M1:** coorte por corte (`Semestre × temporal_marker`). Não é uma medida individual nem uma medida por equipe.

## Os valores do M1 estão corretamente calculados?

### Resposta curta

**Sim, o M1 está aritmeticamente correto.** O recálculo completo, partindo dos 1.122 escores individuais, reproduz exatamente as médias intermediárias e os seis valores publicados. A diferença máxima é zero.

A auditoria abaixo verifica que:

- não existem registros duplicados para a mesma resposta e pergunta;
- não existem escores ausentes;
- todos os escores são inteiros entre 0 e 4;
- todos os cortes possuem as seis perguntas;
- as médias por pergunta conferem com os escores individuais;
- o M1 publicado confere com a média das seis famílias.

Isso confirma a **implementação do cálculo**, mas não prova a **validade da métrica**. Ainda é necessário discutir se o LLM atribui escores confiáveis e se faz sentido combinar as seis perguntas em um único número.

In [50]:
from IPython.display import display

QUESTION_IDS = (
    "ai_benefit",
    "ai_career_impact_5y",
    "autonomy_tool_dependency",
    "career_expectation",
    "project_challenges",
    "project_feeling",
)
MEAN_COLUMNS = [
    f"student_{question}_ai_dependency_score_mean"
    for question in QUESTION_IDS
]
KEYS = ["Semestre", "temporal_marker"]

# Recalcula as médias por pergunta diretamente dos 1.122 escores.
response_means_df = (
    student_nlp_df.groupby(KEYS + ["question_id"])["ai_dependency_score"]
    .mean()
    .unstack("question_id")
)
source_means_df = m1_source_df.set_index(KEYS)[MEAN_COLUMNS].copy()
source_means_df.columns = QUESTION_IDS
max_source_error = (
    response_means_df[list(QUESTION_IDS)] - source_means_df
).abs().to_numpy().max()

# Recalcula o M1 a partir das seis médias e compara com o CSV publicado.
recalculated_df = m1_source_df[KEYS].copy()
recalculated_df["M1_recalculado"] = m1_source_df[MEAN_COLUMNS].mean(axis=1)
recalculated_df["familias_disponiveis"] = m1_source_df[MEAN_COLUMNS].notna().sum(axis=1)
comparison_df = recalculated_df.merge(
    m1_original_df[
        KEYS + ["ai_dependency_composite_mean", "n_question_families_available"]
    ],
    on=KEYS,
    validate="one_to_one",
).rename(columns={"ai_dependency_composite_mean": "M1_publicado"})
comparison_df["diferenca"] = (
    comparison_df["M1_recalculado"] - comparison_df["M1_publicado"]
).abs()

scores = student_nlp_df["ai_dependency_score"]
audit_df = pd.DataFrame(
    [
        {
            "verificação": "Sem resposta-pergunta duplicada",
            "resultado": not student_nlp_df.duplicated(
                ["student_response_id", "question_id"]
            ).any(),
        },
        {
            "verificação": "Sem escores ausentes",
            "resultado": scores.notna().all(),
        },
        {
            "verificação": "Todos os escores são inteiros entre 0 e 4",
            "resultado": scores.between(0, 4).all() and scores.mod(1).eq(0).all(),
        },
        {
            "verificação": "Todos os cortes possuem as seis perguntas",
            "resultado": comparison_df["familias_disponiveis"].eq(6).all()
            and comparison_df["n_question_families_available"].eq(6).all(),
        },
        {
            "verificação": "Médias por pergunta conferem com os escores",
            "resultado": max_source_error < 1e-12,
        },
        {
            "verificação": "M1 publicado confere com o recálculo",
            "resultado": comparison_df["diferenca"].max() < 1e-12,
        },
    ]
)

display(audit_df)
comparison_df.round(6)

,verificação,resultado
0,Sem resposta-pergunta duplicada,True
1,Sem escores ausentes,True
2,Todos os escores são inteiros entre 0 e 4,True
3,Todos os cortes possuem as seis perguntas,True
4,Médias por pergunta conferem com os escores,True
5,M1 publicado confere com o recálculo,True


,Semestre,temporal_marker,M1_recalculado,familias_disponiveis,M1_publicado,n_question_families_available,diferenca
0,2025.2,T1,1.210145,6,1.210145,6,0.0
1,2025.2,T2,1.300813,6,1.300813,6,0.0
2,2025.2,T3,1.283333,6,1.283333,6,0.0
3,2026.1,T1,1.159091,6,1.159091,6,0.0
4,2026.1,T2,1.308333,6,1.308333,6,0.0
5,2026.1,T3,1.324074,6,1.324074,6,0.0


## Que outros indicadores poderiam responder melhor à RQ1?

### Resposta curta

O M1 atual **não responde sozinho à RQ1**. Ele aplica o mesmo escore de “dependência de IA” a seis perguntas com objetivos diferentes.

Além disso, o survey atual não mede diretamente o uso real de IA:

- benefício da IA pode ser **obtido ou imaginado**;
- autonomia pergunta qual seria o **equilíbrio ideal**, não a dependência real;
- impacto da IA em cinco anos mede percepção futura de carreira;
- expectativa de carreira é geral e não exige menção à IA;
- desafios e sentimentos do projeto não são perguntas sobre IA.

A alternativa mais simples é usar um painel:

| Indicador | O que responde | Situação |
|---|---|---|
| Uso real da IA | Se usa IA, frequência e tipos de tarefa | Não está medido diretamente; exige novas perguntas ou codificação de evidência explícita de uso. |
| Equilíbrio ideal entre autonomia e ferramentas | Preferência declarada sobre autonomia | Pode usar a pergunta atual, mas não deve ser chamado de dependência real. |
| Impacto percebido na carreira | Como a IA pode alterar competências, tarefas e riscos | A pergunta de impacto em cinco anos pode ser recodificada com categorias específicas. |
| Expectativas do projeto | Desafios, sentimento e carga percebida | Pode usar as perguntas atuais, separadas dos indicadores de IA. |

O resultado principal deve mostrar esses indicadores separadamente por semestre e corte, sem calcular uma média entre eles.

### O que a RQ1 exige

> **RQ1:** How do student developers utilize Generative AI, and how does this adoption shape their longitudinal perceptions of software engineering roles and project expectations?

A RQ1 possui três partes diferentes:

1. **uso da IA**: se, como e para quais tarefas ela é usada;
2. **percepção dos papéis profissionais**: como os estudantes esperam que a IA altere atividades e competências;
3. **expectativas do projeto**: desafios, sentimentos e carga percebida ao longo de T1, T2 e T3.

Um indicador é adequado quando responde diretamente a uma dessas partes. Juntar as três em uma média dificulta a interpretação.

### O que os dados atuais permitem medir

Sem novas chamadas ao LLM, os dados atuais permitem apenas uma **análise exploratória**:

- distribuição dos escores atuais por pergunta;
- preferência declarada sobre autonomia e ferramentas;
- percepção do impacto da IA na carreira em cinco anos;
- desafios, sentimento e carga percebida sobre o projeto;
- diferenças descritivas entre T1 e T3 para cada pergunta.

Eles não permitem afirmar diretamente:

- prevalência ou frequência real de uso da IA;
- tarefas em que a IA foi efetivamente usada;
- dependência real do estudante;
- mudança individual entre T1 e T3.

Os dados de Git atuais também não identificam uso de IA. Portanto, não devem ser usados como evidência direta de adoção.

In [51]:
QUESTION_FAMILIES = {
    "ai_benefit": "Benefício percebido da IA",
    "ai_career_impact_5y": "Impacto da IA na carreira em cinco anos",
    "autonomy_tool_dependency": "Autonomia e dependência de ferramentas",
    "career_expectation": "Expectativa de carreira",
    "project_challenges": "Desafios esperados no projeto",
    "project_feeling": "Sentimento sobre o projeto",
}

records = []
for _, row in m1_source_df.iterrows():
    for family, label in QUESTION_FAMILIES.items():
        prefix = f"student_{family}_ai_dependency_score"
        records.append(
            {
                "Semestre": row["Semestre"],
                "temporal_marker": row["temporal_marker"],
                "family": family,
                "prompt_family": label,
                "mean_0_4": row[f"{prefix}_mean"],
                "std_0_4": row[f"{prefix}_std"],
                "median_0_4": row[f"{prefix}_median"],
                "iqr_0_4": row[f"{prefix}_iqr"],
                "n": row[f"{prefix}_n_valid"],
            }
        )

family_summary_df = pd.DataFrame(records)
family_summary_df

,Semestre,temporal_marker,family,prompt_family,mean_0_4,std_0_4,median_0_4,iqr_0_4,n
0,2025.2,T1,ai_benefit,Benefício percebido da IA,2.130435,1.002413,2.0,1.00,46
1,2025.2,T1,ai_career_impact_5y,Impacto da IA na carreira em cinco anos,1.478261,1.005300,1.0,1.00,46
2,2025.2,T1,autonomy_tool_dependency,Autonomia e dependência de ferramentas,3.086957,0.890096,3.0,2.00,46
3,2025.2,T1,career_expectation,Expectativa de carreira,0.195652,0.542405,0.0,0.00,46
4,2025.2,T1,project_challenges,Desafios esperados no projeto,0.260870,0.491473,0.0,0.00,46
5,2025.2,T1,project_feeling,Sentimento sobre o projeto,0.108696,0.433501,0.0,0.00,46
6,2025.2,T2,ai_benefit,Benefício percebido da IA,2.219512,0.758689,2.0,1.00,41
7,2025.2,T2,ai_career_impact_5y,Impacto da IA na carreira em cinco anos,1.439024,1.025885,1.0,1.00,41
8,2025.2,T2,autonomy_tool_dependency,Autonomia e dependência de ferramentas,3.024390,0.851111,3.0,2.00,41
9,2025.2,T2,career_expectation,Expectativa de carreira,0.634146,0.887584,0.0,1.00,41


### Por que manter as seis perguntas separadas?

A média única só seria adequada se as seis perguntas medissem o mesmo construto e tivessem interpretação comparável. Isso não foi demonstrado.

Além disso, uma resposta sobre sentimentos ou desafios do projeto pode receber escore zero simplesmente porque não menciona IA. Nesse caso, zero significa “sem evidência de dependência nesta pergunta”, e não necessariamente “estudante sem dependência”. Incluir esse zero na média reduz artificialmente o M1.

Por isso, a análise principal deve mostrar a trajetória de cada pergunta separadamente. Um resumo combinado pode ser mantido apenas como análise de sensibilidade, claramente identificado como não validado.

In [52]:
import plotly.express as px
from IPython.display import HTML, display

family_trajectory_figure = px.line(
    family_summary_df,
    x="temporal_marker",
    y="mean_0_4",
    color="prompt_family",
    facet_col="Semestre",
    markers=True,
    category_orders={"temporal_marker": ["T1", "T2", "T3"]},
    labels={
        "temporal_marker": "Corte",
        "mean_0_4": "Média do escore atual (0–4)",
        "prompt_family": "Pergunta",
    },
    title="Escores atuais separados por pergunta",
)
family_trajectory_figure.update_yaxes(range=[0, 4])
family_trajectory_figure.update_layout(legend_title_text="Pergunta")
display(HTML(family_trajectory_figure.to_html(include_plotlyjs="cdn", full_html=False)))

In [53]:
family_profile_df = family_summary_df.pivot(
    index="prompt_family",
    columns=["Semestre", "temporal_marker"],
    values="mean_0_4",
)
family_profile_df.round(3)

Semestre                                2025.2               2026.1        \
temporal_marker                             T1     T2     T3     T1    T2   
prompt_family                                                               
Autonomia e dependência de ferramentas   3.087  3.024  3.225  2.636  2.90   
Benefício percebido da IA                2.130  2.220  2.250  2.000  1.85   
Desafios esperados no projeto            0.261  0.171  0.150  0.136  0.20   
Expectativa de carreira                  0.196  0.634  0.400  0.318  0.55   
Impacto da IA na carreira em cinco anos  1.478  1.439  1.325  1.545  1.80   
Sentimento sobre o projeto               0.109  0.317  0.350  0.318  0.55   

Semestre                                        
temporal_marker                             T3  
prompt_family                                   
Autonomia e dependência de ferramentas   3.444  
Benefício percebido da IA                1.556  
Desafios esperados no projeto            0.056  
Expectativa de carreira                  0.722  
Impacto da IA na carreira em cinco anos  1.667  
Sentimento sobre o projeto               0.500

### A escala deve mudar de 0–4 para 0–10?

**Não.** Trocar apenas a faixa numérica não melhora a métrica. A transformação $x_{0–10}=2{,}5x_{0–4}$ preserva a ordem e as diferenças relativas; ela apenas muda a apresentação.

O problema real é que a rubrica atual pede um inteiro de 0 a 4, mas não define claramente o significado de cada categoria. A refatoração deve:

1. definir âncoras observáveis para cada categoria;
2. usar categorias específicas para cada indicador;
3. validar uma amostra com codificação humana e LLM;
4. medir concordância antes de processar todo o corpus.

A escala 0–4 pode ser mantida se essas condições forem atendidas. Uma nova escala 0–10 seria um novo instrumento e exigiria a mesma validação.

In [54]:
scale_check_df = family_summary_df[
    ["Semestre", "temporal_marker", "prompt_family", "mean_0_4"]
].copy()
scale_check_df["mean_0_10"] = scale_check_df["mean_0_4"] * 2.5

scale_check = pd.Series(
    {
        "correlação entre as escalas": scale_check_df[
            ["mean_0_4", "mean_0_10"]
        ].corr().iloc[0, 1],
        "erro máximo da transformação": (
            scale_check_df["mean_0_10"] - 2.5 * scale_check_df["mean_0_4"]
        ).abs().max(),
        "mesma ordenação": scale_check_df["mean_0_4"].rank().equals(
            scale_check_df["mean_0_10"].rank()
        ),
    },
    name="resultado",
)
scale_check

correlação entre as escalas      1.0
erro máximo da transformação     0.0
mesma ordenação                 True
Name: resultado, dtype: object

### Por que apresentar mais do que a média?

A média mostra apenas o centro da distribuição. Para escores ordinais, cada indicador deve incluir:

- **mediana e IQR:** resumo principal da posição e dispersão;
- **proporção em cada categoria:** mostra concentração e polarização;
- **média e desvio-padrão:** resumo complementar, caso as categorias sejam tratadas como igualmente espaçadas;
- **tamanho da amostra:** torna perdas entre cortes visíveis.

Não se deve calcular o desvio-padrão do M1 usando apenas os desvios das seis perguntas. Isso exigiria os escores por estudante e a covariância entre perguntas.

Também não se deve chamar $T3-T1$ de mudança individual. Com os dados atuais, essa diferença representa uma **mudança descritiva entre coortes observadas em cada corte**. Uma análise longitudinal individual exige identificar os mesmos estudantes em T1 e T3.

In [55]:
descriptive_summary_df = family_summary_df[
    [
        "Semestre",
        "temporal_marker",
        "prompt_family",
        "n",
        "mean_0_4",
        "std_0_4",
        "median_0_4",
        "iqr_0_4",
    ]
].copy()
descriptive_summary_df.round(3)

,Semestre,temporal_marker,prompt_family,n,mean_0_4,std_0_4,median_0_4,iqr_0_4
0,2025.2,T1,Benefício percebido da IA,46,2.130,1.002,2.0,1.00
1,2025.2,T1,Impacto da IA na carreira em cinco anos,46,1.478,1.005,1.0,1.00
2,2025.2,T1,Autonomia e dependência de ferramentas,46,3.087,0.890,3.0,2.00
3,2025.2,T1,Expectativa de carreira,46,0.196,0.542,0.0,0.00
4,2025.2,T1,Desafios esperados no projeto,46,0.261,0.491,0.0,0.00
5,2025.2,T1,Sentimento sobre o projeto,46,0.109,0.434,0.0,0.00
6,2025.2,T2,Benefício percebido da IA,41,2.220,0.759,2.0,1.00
7,2025.2,T2,Impacto da IA na carreira em cinco anos,41,1.439,1.026,1.0,1.00
8,2025.2,T2,Autonomia e dependência de ferramentas,41,3.024,0.851,3.0,2.00
9,2025.2,T2,Expectativa de carreira,41,0.634,0.888,0.0,1.00


In [56]:
family_change_df = (
    family_summary_df.pivot(
        index=["Semestre", "prompt_family"],
        columns="temporal_marker",
        values="mean_0_4",
    )
    .assign(mudanca_T1_T3=lambda frame: frame["T3"] - frame["T1"])
    .reset_index()
)
family_change_df.round(3)

temporal_marker,Semestre,prompt_family,T1,T2,T3,mudanca_T1_T3
0,2025.2,Autonomia e dependência de ferramentas,3.087,3.024,3.225,0.138
1,2025.2,Benefício percebido da IA,2.130,2.220,2.250,0.120
2,2025.2,Desafios esperados no projeto,0.261,0.171,0.150,-0.111
3,2025.2,Expectativa de carreira,0.196,0.634,0.400,0.204
4,2025.2,Impacto da IA na carreira em cinco anos,1.478,1.439,1.325,-0.153
5,2025.2,Sentimento sobre o projeto,0.109,0.317,0.350,0.241
6,2026.1,Autonomia e dependência de ferramentas,2.636,2.900,3.444,0.808
7,2026.1,Benefício percebido da IA,2.000,1.850,1.556,-0.444
8,2026.1,Desafios esperados no projeto,0.136,0.200,0.056,-0.081
9,2026.1,Expectativa de carreira,0.318,0.550,0.722,0.404


### Decisão de refatoração

Substituir o M1 único por quatro indicadores independentes:

1. **M1a — Uso real da IA:** prevalência, frequência e tipos de tarefa. Requer novas perguntas fechadas ou codificação de evidência textual explícita de uso.
2. **M1b — Equilíbrio ideal entre autonomia e ferramentas:** distribuição da preferência declarada. Pode reutilizar a pergunta atual após definir uma rubrica própria; não mede dependência real.
3. **M1c — Impacto percebido na carreira:** proporção de temas sobre produtividade, competências, substituição de tarefas, risco e incerteza. Pode recodificar a pergunta específica sobre o impacto da IA em cinco anos.
4. **M1d — Expectativas do projeto:** desafios, sentimento e carga percebida. Deve usar rubricas próprias e permanecer separado dos indicadores de IA.

O M1 atual deve permanecer somente como **baseline legado** para comparar os efeitos da refatoração. Ele não deve ser o resultado principal da RQ1.

#### Critérios de aceite

A refatoração estará pronta quando:

- cada indicador estiver ligado explicitamente a uma parte da RQ1;
- cada categoria tiver definição e exemplos de inclusão/exclusão;
- uma amostra tiver sido codificada por humanos e pelo LLM;
- a concordância tiver sido medida e considerada adequada;
- resultados mostrarem distribuições e tamanho da amostra, não apenas médias;
- comparações principais forem separadas por semestre e corte;
- qualquer análise individual usar identificadores estáveis entre T1 e T3.

### Ordem de implementação

1. Congelar o CSV atual como `M1 legado`.
2. Definir o contrato de saída de M1a–M1d e as categorias de cada indicador.
3. Criar um conjunto pequeno de respostas codificadas manualmente.
4. Testar e revisar as rubricas até obter concordância adequada.
5. Gerar resultados no nível `resposta × indicador`, preservando semestre e corte.
6. Agregar distribuições por `Semestre × temporal_marker`.
7. Comparar os novos indicadores com o baseline legado e documentar as diferenças.
8. Atualizar texto, tabelas e figuras do artigo somente após essa validação.

### Agrupar por semestre ou apenas por T1, T2 e T3?

Cada semestre contém uma coorte diferente. Por isso, a análise principal deve manter `Semestre × temporal_marker`.

Agrupar apenas por T1, T2 e T3 responde outra pergunta: qual é a média entre as coortes observadas em cada corte? Esse resumo pode esconder diferenças entre semestres e dar maior peso à coorte com mais respostas.

Duas formas de agregação são possíveis:

- **ponderada por estudantes:** a coorte maior recebe mais peso;
- **peso igual por semestre:** cada coorte recebe o mesmo peso.

O agrupamento combinado pode ser apresentado como resumo secundário. Ele não substitui as trajetórias por semestre e não transforma os dados em acompanhamento longitudinal individual.

In [57]:
m1_by_semester_df = m1_original_df.merge(
    m1_source_df[["Semestre", "temporal_marker", "student_n"]],
    on=["Semestre", "temporal_marker"],
    validate="one_to_one",
)

pooled_m1_df = (
    m1_by_semester_df.groupby("temporal_marker", as_index=False)
    .apply(
        lambda group: pd.Series(
            {
                "student_weighted_mean": (
                    group["ai_dependency_composite_mean"] * group["student_n"]
                ).sum()
                / group["student_n"].sum(),
                "equal_semester_mean": group[
                    "ai_dependency_composite_mean"
                ].mean(),
                "total_students": group["student_n"].sum(),
                "semester_min": group["ai_dependency_composite_mean"].min(),
                "semester_max": group["ai_dependency_composite_mean"].max(),
                "semester_gap": group["ai_dependency_composite_mean"].max()
                - group["ai_dependency_composite_mean"].min(),
            }
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)

pooled_m1_df.round(3)

,temporal_marker,student_weighted_mean,equal_semester_mean,total_students,semester_min,semester_max,semester_gap
0,T1,1.194,1.185,68.0,1.159,1.210,0.051
1,T2,1.303,1.305,61.0,1.301,1.308,0.008
2,T3,1.296,1.304,58.0,1.283,1.324,0.041


In [58]:
import plotly.graph_objects as go

semester_plot_df = m1_by_semester_df[
    ["Semestre", "temporal_marker", "ai_dependency_composite_mean"]
].rename(
    columns={
        "Semestre": "serie",
        "ai_dependency_composite_mean": "M1_mean",
    }
)

pooled_plot_df = pd.concat(
    [
        pooled_m1_df[
            ["temporal_marker", "student_weighted_mean"]
        ].rename(columns={"student_weighted_mean": "M1_mean"}).assign(
            serie="Agregado: ponderado por estudantes"
        ),
        pooled_m1_df[
            ["temporal_marker", "equal_semester_mean"]
        ].rename(columns={"equal_semester_mean": "M1_mean"}).assign(
            serie="Agregado: peso igual por semestre"
        ),
    ],
    ignore_index=True,
)

semester_pooling_figure = go.Figure()
for series_name, series_df in pd.concat(
    [semester_plot_df, pooled_plot_df],
    ignore_index=True,
).groupby("serie", sort=False):
    semester_pooling_figure.add_trace(
        go.Scatter(
            x=series_df["temporal_marker"],
            y=series_df["M1_mean"],
            mode="lines+markers",
            name=series_name,
        )
    )

semester_pooling_figure.update_layout(
    title="M1 legado por semestre e agregado",
    xaxis_title="Corte",
    yaxis_title="Média do M1 legado (0–4)",
    yaxis_range=[0, 4],
    legend_title_text="Agrupamento",
)
display(
    HTML(
        semester_pooling_figure.to_html(
            include_plotlyjs="cdn",
            full_html=False,
        )
    )
)

#### Resultado nos dados atuais

Neste conjunto, agregar os semestres quase não altera o M1 legado:

- diferença entre semestres: 0,051 em T1; 0,008 em T2; 0,041 em T3;
- agregado ponderado: 1,194 em T1; 1,303 em T2; 1,296 em T3;
- agregado com pesos iguais: 1,185 em T1; 1,305 em T2; 1,304 em T3.

Portanto, o agregado pode aparecer como resumo secundário. As linhas por semestre devem continuar visíveis para demonstrar que essa semelhança foi observada, e não presumida.

Esse teste usa apenas o **M1 legado**. Após a refatoração, ele deve ser repetido separadamente para M1a–M1d.

## Resumo das mudanças propostas

A prioridade é responder corretamente à RQ1, mesmo que isso exija alterar contratos, prompts e artefatos e executar novamente parte da pipeline.

| Aspecto | Situação atual | Mudança proposta | Regeneração |
|---|---|---|---|
| Estrutura do M1 | Uma média de seis perguntas diferentes | Quatro indicadores independentes: M1a–M1d | Sim |
| Uso real da IA | Não é medido diretamente | Codificar apenas evidência explícita de uso, frequência e tarefa; separar uso real de uso imaginado | Novas chamadas LLM; nova coleta pode ser necessária |
| Autonomia | Tratada como dependência real | Medir preferência pelo equilíbrio ideal entre autonomia e ferramentas | Novas chamadas LLM com rubrica própria |
| Carreira | Escore genérico de dependência | Codificar produtividade, competências, substituição, risco e incerteza | Novas chamadas LLM com taxonomia própria |
| Projeto | Misturado com dependência de IA | Manter desafios, sentimento e carga como indicadores separados | Recomendado recodificar com rubricas próprias |
| Escala | Mesmo 0–4 sem âncoras para todas as perguntas | Categorias ancoradas e específicas para cada indicador | Sim |
| Estatística | Ênfase na média | Proporções por categoria, mediana, IQR, média complementar e $n$ | Recalcular agregados |
| Tempo | Comparação de cortes chamada de trajetória | Separar semestre e corte; usar “longitudinal individual” somente com vínculo entre estudantes | Recalcular agregados |
| M1 legado | Resultado principal | Baseline de comparação e análise de sensibilidade | Preservar sem sobrescrever |

### Decisão sobre a pipeline

**É recomendado regenerar os dados de NLP e todos os artefatos dependentes do M1.** Não é necessário regenerar dados brutos, Git ou transcrições que não participam desses novos indicadores.

A execução deve seguir esta ordem:

1. preservar os artefatos atuais como versão legada;
2. definir e versionar contratos, taxonomias e prompts de M1a–M1d;
3. criar uma amostra de referência codificada manualmente;
4. testar as novas chamadas LLM nessa amostra;
5. revisar as rubricas até obter concordância adequada;
6. executar novamente a etapa de NLP sobre as respostas originais;
7. regenerar agregados, análises, tabelas e figuras dependentes;
8. comparar a versão nova com o M1 legado e documentar diferenças.

Não devemos executar novas chamadas LLM antes das etapas 2–5. Caso contrário, apenas aumentaremos custo sem resolver o problema de validade.

### Limite dos dados existentes

Uma nova pipeline não cria informação ausente. Como o survey permite respostas sobre benefícios imaginados e não pergunta diretamente frequência e tarefas de uso, M1a deve distinguir:

- uso real explicitamente relatado;
- uso apenas imaginado ou hipotético;
- ausência de evidência;
- resposta insuficiente.

Se a evidência explícita for escassa, a resposta adequada não é inferir uso. Será necessário coletar novas respostas com perguntas diretas sobre frequência, ferramentas e tarefas.